#Part 1

In [108]:
!pip install pyspark

In [109]:
import pyspark
from pyspark.sql import SparkSession

print("PySpark Version:", pyspark.__version__)

PySpark Version: 4.0.3


In [110]:
spark = SparkSession.builder \
    .appName("NYC Taxi Analytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [111]:
import platform
import sys

print("Python Version:", sys.version)
print("Operating System:", platform.system())
print("Spark Version:", spark.version)


Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Operating System: Linux
Spark Version: 4.0.3


In [112]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [113]:
print("Spark Configuration")

for key, value in spark.sparkContext.getConf().getAll():
    print(f"{key}: {value}")

Spark Configuration
spark.rdd.compress: True
spark.hadoop.fs.s3a.vectored.read.min.seek.size: 128K
spark.app.startTime: 1785675883155
spark.executor.extraJavaOptions: -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.k

#Part 2

In [114]:
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

In [115]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [116]:
print("Number of Columns:", len(df.columns))

Number of Columns: 19


In [117]:
print("Number of Records:", df.count())

Number of Records: 2964624


In [118]:
df.show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [119]:
print(df.columns)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


#Part 3


In [120]:
print("Total Trips:", df.count())

Total Trips: 2964624


In [121]:
from pyspark.sql.functions import min

df.select(min("tpep_pickup_datetime").alias("Earliest_Trip_Date")).show()

+-------------------+
| Earliest_Trip_Date|
+-------------------+
|2002-12-31 22:59:39|
+-------------------+



In [122]:
from pyspark.sql.functions import max

df.select(max("tpep_pickup_datetime").alias("Latest_Trip_Date")).show()

+-------------------+
|   Latest_Trip_Date|
+-------------------+
|2024-02-01 00:01:15|
+-------------------+



In [123]:
df.select("VendorID").distinct().show()

print("Number of Unique Vendors:",
      df.select("VendorID").distinct().count())

+--------+
|VendorID|
+--------+
|       1|
|       2|
|       6|
+--------+

Number of Unique Vendors: 3


In [124]:
from pyspark.sql.functions import avg

df.select(avg("trip_distance")).show()

+------------------+
|avg(trip_distance)|
+------------------+
|3.6521691789580624|
+------------------+



In [125]:
df.select(avg("fare_amount")).show()

+------------------+
|  avg(fare_amount)|
+------------------+
|18.175061916792536|
+------------------+



In [126]:
from pyspark.sql.functions import max

df.select(max("fare_amount")).show()

+----------------+
|max(fare_amount)|
+----------------+
|          5000.0|
+----------------+



In [127]:
from pyspark.sql.functions import min

df.select(min("fare_amount")).show()

+----------------+
|min(fare_amount)|
+----------------+
|          -899.0|
+----------------+



In [128]:
df.select(avg("passenger_count")).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.3392808966805005|
+--------------------+



In [129]:
df.select("payment_type").distinct().show()

print("Number of Payment Methods:",
      df.select("payment_type").distinct().count())

+------------+
|payment_type|
+------------+
|           1|
|           3|
|           2|
|           4|
|           0|
+------------+

Number of Payment Methods: 5


#Part 4 – Data Cleaning

In [130]:
print("Original Number of Records:", df.count())

Original Number of Records: 2964624


In [131]:
df = df.dropDuplicates()

print("Number of Records After Removing Duplicates:")
print(df.count())

Number of Records After Removing Duplicates:
2964624


In [132]:
df = df.filter(df.trip_distance > 0)

print("Records After Removing Invalid Trip Distances:")
print(df.count())

Records After Removing Invalid Trip Distances:
2904253


In [133]:
df = df.filter(df.fare_amount >= 0)

print("Records After Removing Negative Fares:")
print(df.count())

Records After Removing Negative Fares:
2870188


In [134]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=True)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 115293 
 trip_distance         | 0      
 RatecodeID            | 115293 
 store_and_fwd_flag    | 115293 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 115293 
 Airport_fee           | 115293 



In [135]:
df = df.dropna()

print("Records After Removing Missing Values:")
print(df.count())

Records After Removing Missing Values:
2754895


In [136]:
df = df.fillna(0)

In [137]:
print("Final Number of Records:", df.count())

df.show(10, truncate=False)

Final Number of Records: 2754895
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:47:21 |2024-01-01 00:51:58  |2              |1.13         |1         |N                 |238         |166         |1           |7.

#Part 5 – Spark Transformations

In [138]:
high_fare = df.filter(df.fare_amount > 50)

high_fare.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 01:41:09|  2024-01-01 02:07:52|              1|        19.69|         5|                 N|         262|         265|           1|      100.0|  0.0|    0.0|      20.

In [139]:
selected = df.select("VendorID","passenger_count","trip_distance","fare_amount")
selected.show(10)

+--------+---------------+-------------+-----------+
|VendorID|passenger_count|trip_distance|fare_amount|
+--------+---------------+-------------+-----------+
|       2|              2|         1.13|        7.9|
|       1|              3|          0.8|        6.5|
|       2|              1|         4.43|       31.0|
|       2|              1|         3.64|       17.7|
|       2|              2|          5.9|       30.3|
|       2|              2|         3.43|       24.7|
|       2|              1|         0.95|        8.6|
|       2|              2|         1.83|       10.7|
|       2|              1|         2.25|       14.2|
|       1|              2|          0.8|        7.2|
+--------+---------------+-------------+-----------+
only showing top 10 rows


In [140]:
from pyspark.sql.functions import round

fare_per_mile = df.withColumn(
    "fare_per_mile",
    round(df.fare_amount / df.trip_distance, 2)
)

fare_per_mile.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|fare_per_mile|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------+
|       2| 2024-01-01 00:47:21|  2024-01-01 00:51:58|              2|         1.13|         1|                 N|         238|         166|      

In [141]:
sorted_df = df.orderBy(df.fare_amount.desc())

sorted_df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-14 10:08:11|  2024-01-16 13:54:22|              1|        31.95|         1|                 N|         220|         220|           2|     2221.3|  0.0|    0.5|       0.

In [142]:
dropped = df.drop("store_and_fwd_flag")

dropped.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:47:21|  2024-01-01 00:51:58|              2|         1.13|         1|         238|         166|           1|        7.9|  1.0|    0.5|      2.58|         0.0|                  1.0|       15.48|                 2.5|     

In [143]:
payment_methods = df.select("payment_type").distinct()

payment_methods.show()

+------------+
|payment_type|
+------------+
|           1|
|           3|
|           2|
|           4|
+------------+



In [144]:
from pyspark.sql.functions import avg

grouped = df.groupBy("payment_type").agg(
    avg("fare_amount").alias("Average_Fare")
)

grouped.show()

+------------+------------------+
|payment_type|      Average_Fare|
+------------+------------------+
|           1|18.376087258997696|
|           3|17.040781294018217|
|           2|18.613623047914025|
|           4| 19.66531404344596|
+------------+------------------+



In [145]:
payment_df = df.select("payment_type").distinct()

joined = payment_df.join(
    grouped,
    on="payment_type",
    how="inner"
)

joined.show()

+------------+------------------+
|payment_type|      Average_Fare|
+------------+------------------+
|           1|18.376087258997696|
|           3|17.040781294018217|
|           2|18.613623047914025|
|           4| 19.66531404344596|
+------------+------------------+



In [146]:
alias_df = df.select(
    df.VendorID.alias("Vendor"),
    df.trip_distance.alias("Distance"),
    df.fare_amount.alias("Fare")
)

alias_df.show(10)

+------+--------+----+
|Vendor|Distance|Fare|
+------+--------+----+
|     2|    1.13| 7.9|
|     1|     0.8| 6.5|
|     2|    4.43|31.0|
|     2|    3.64|17.7|
|     2|     5.9|30.3|
|     2|    3.43|24.7|
|     2|    0.95| 8.6|
|     2|    1.83|10.7|
|     2|    2.25|14.2|
|     1|     0.8| 7.2|
+------+--------+----+
only showing top 10 rows


In [147]:
repartitioned = df.repartition(4)

print("Partitions:", repartitioned.rdd.getNumPartitions())

Partitions: 4


#Part 6 – Spark SQL

In [148]:
df.createOrReplaceTempView("taxi_trips")

In [149]:
spark.sql("""SELECT trip_distance,fare_amount,passenger_count FROM taxi_trips ORDER BY trip_distance DESC LIMIT 10 """).show()

+-------------+-----------+---------------+
|trip_distance|fare_amount|passenger_count|
+-------------+-----------+---------------+
|     15400.32|       28.9|              1|
|     10879.28|       70.0|              1|
|      1715.22|       70.0|              1|
|        971.8|       21.5|              1|
|        964.6|       39.5|              1|
|        277.4|       33.8|              2|
|       246.22|        8.6|              1|
|       233.25|     1616.5|              1|
|       210.82|      500.0|              1|
|        210.2|      650.0|              1|
+-------------+-----------+---------------+



In [150]:
spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY PULocationID
ORDER BY Total_Trips DESC
LIMIT 10
""").show()

+------------+-----------+
|PULocationID|Total_Trips|
+------------+-----------+
|         132|     138130|
|         237|     137093|
|         161|     136500|
|         236|     129604|
|         162|     102308|
|         186|     100737|
|         230|     100008|
|         142|      98906|
|         138|      87253|
|         239|      82391|
+------------+-----------+



In [151]:
spark.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount),2) AS Average_Fare
FROM taxi_trips
GROUP BY payment_type
ORDER BY payment_type
""").show()

+------------+------------+
|payment_type|Average_Fare|
+------------+------------+
|           1|       18.38|
|           2|       18.61|
|           3|       17.04|
|           4|       19.67|
+------------+------------+



In [152]:
spark.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS Pickup_Hour,
    COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY Pickup_Hour
ORDER BY Total_Trips DESC
LIMIT 1
""").show()

+-----------+-----------+
|Pickup_Hour|Total_Trips|
+-----------+-----------+
|         18|     198037|
+-----------+-----------+



In [153]:
spark.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount
FROM taxi_trips
WHERE trip_distance > 20
ORDER BY trip_distance DESC
""").show(10)

+--------+-------------+-----------+
|VendorID|trip_distance|fare_amount|
+--------+-------------+-----------+
|       2|     15400.32|       28.9|
|       2|     10879.28|       70.0|
|       2|      1715.22|       70.0|
|       1|        971.8|       21.5|
|       1|        964.6|       39.5|
|       2|        277.4|       33.8|
|       2|       246.22|        8.6|
|       2|       233.25|     1616.5|
|       2|       210.82|      500.0|
|       1|        210.2|      650.0|
+--------+-------------+-----------+
only showing top 10 rows


In [154]:
spark.sql("""
SELECT
    MONTH(tpep_pickup_datetime) AS Month,
    ROUND(SUM(total_amount),2) AS Revenue
FROM taxi_trips
GROUP BY Month
ORDER BY Month
""").show()

+-----+-------------+
|Month|      Revenue|
+-----+-------------+
|    1|7.542614842E7|
|    2|        90.47|
|   12|       235.12|
+-----+-------------+



In [155]:
spark.sql("""
SELECT
    VendorID,
    ROUND(AVG(trip_distance),2) AS Avg_Distance
FROM taxi_trips
GROUP BY VendorID
""").show()

+--------+------------+
|VendorID|Avg_Distance|
+--------+------------+
|       1|        3.13|
|       2|        3.35|
+--------+------------+



In [156]:
spark.sql("""
SELECT
    VendorID,
    fare_amount
FROM taxi_trips
ORDER BY fare_amount DESC
LIMIT 10
""").show()

+--------+-----------+
|VendorID|fare_amount|
+--------+-----------+
|       2|     2221.3|
|       2|     1616.5|
|       2|      912.3|
|       2|      899.0|
|       2|      820.0|
|       2|      761.1|
|       2|      749.2|
|       2|      744.3|
|       2|      739.4|
|       2|      700.0|
+--------+-----------+



In [157]:
spark.sql("""
SELECT
    ROUND(AVG(passenger_count),2) AS Average_Passengers
FROM taxi_trips
""").show()

+------------------+
|Average_Passengers|
+------------------+
|              1.34|
+------------------+



In [158]:
spark.sql("""
SELECT
    payment_type,
    COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY payment_type
ORDER BY Total_Trips DESC
""").show()

+------------+-----------+
|payment_type|Total_Trips|
+------------+-----------+
|           1|    2298422|
|           2|     422945|
|           4|      22879|
|           3|      10649|
+------------+-----------+



#Part 7 – Window Functions

In [159]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

In [160]:
windowSpec = Window.orderBy(df.fare_amount.desc())

In [161]:
row_num_df = df.withColumn(
    "row_number",
    row_number().over(windowSpec)
)

row_num_df.select(
    "VendorID",
    "fare_amount",
    "row_number"
).show(10)

+--------+-----------+----------+
|VendorID|fare_amount|row_number|
+--------+-----------+----------+
|       2|     2221.3|         1|
|       2|     1616.5|         2|
|       2|      912.3|         3|
|       2|      899.0|         4|
|       2|      820.0|         5|
|       2|      761.1|         6|
|       2|      749.2|         7|
|       2|      744.3|         8|
|       2|      739.4|         9|
|       2|      700.0|        10|
+--------+-----------+----------+
only showing top 10 rows


In [162]:
rank_df = df.withColumn(
    "rank",
    rank().over(windowSpec)
)

rank_df.select(
    "VendorID",
    "fare_amount",
    "rank"
).show(10)

+--------+-----------+----+
|VendorID|fare_amount|rank|
+--------+-----------+----+
|       2|     2221.3|   1|
|       2|     1616.5|   2|
|       2|      912.3|   3|
|       2|      899.0|   4|
|       2|      820.0|   5|
|       2|      761.1|   6|
|       2|      749.2|   7|
|       2|      744.3|   8|
|       2|      739.4|   9|
|       2|      700.0|  10|
+--------+-----------+----+
only showing top 10 rows


In [163]:
dense_rank_df = df.withColumn(
    "dense_rank",
    dense_rank().over(windowSpec)
)

dense_rank_df.select(
    "VendorID",
    "fare_amount",
    "dense_rank"
).show(10)

+--------+-----------+----------+
|VendorID|fare_amount|dense_rank|
+--------+-----------+----------+
|       2|     2221.3|         1|
|       2|     1616.5|         2|
|       2|      912.3|         3|
|       2|      899.0|         4|
|       2|      820.0|         5|
|       2|      761.1|         6|
|       2|      749.2|         7|
|       2|      744.3|         8|
|       2|      739.4|         9|
|       2|      700.0|        10|
+--------+-----------+----------+
only showing top 10 rows


#Part 8 – Performance Optimization

In [164]:
import time

In [165]:
start_time = time.time()

df.groupBy("payment_type").count().show()

end_time = time.time()

print("Execution Time Before Caching:", end_time - start_time, "seconds")

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2298422|
|           3|  10649|
|           2| 422945|
|           4|  22879|
+------------+-------+

Execution Time Before Caching: 2.351300001144409 seconds


In [166]:
df.cache()

# Materialize the cache
df.count()

print("Dataset cached successfully!")

Dataset cached successfully!


In [167]:
start_time = time.time()

df.groupBy("payment_type").count().show()

end_time = time.time()

print("Execution Time After Caching:", end_time - start_time, "seconds")

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2298422|
|           3|  10649|
|           2| 422945|
|           4|  22879|
+------------+-------+

Execution Time After Caching: 4.002077341079712 seconds


In [168]:
print("Partitions Before:", df.rdd.getNumPartitions())

repartitioned_df = df.repartition(4)

print("Partitions After:", repartitioned_df.rdd.getNumPartitions())

Partitions Before: 200
Partitions After: 4


In [169]:
df.groupBy("payment_type").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['payment_type], ['payment_type, 'count(1) AS count#25466]
+- Project [coalesce(VendorID#10813, cast(0.0 as int)) AS VendorID#11343, tpep_pickup_datetime#10814, tpep_dropoff_datetime#10815, coalesce(passenger_count#10816L, cast(0.0 as bigint)) AS passenger_count#11344L, coalesce(nanvl(trip_distance#10817, cast(null as double)), cast(0.0 as double)) AS trip_distance#11345, coalesce(RatecodeID#10818L, cast(0.0 as bigint)) AS RatecodeID#11346L, store_and_fwd_flag#10819, coalesce(PULocationID#10820, cast(0.0 as int)) AS PULocationID#11347, coalesce(DOLocationID#10821, cast(0.0 as int)) AS DOLocationID#11348, coalesce(payment_type#10822L, cast(0.0 as bigint)) AS payment_type#11349L, coalesce(nanvl(fare_amount#10823, cast(null as double)), cast(0.0 as double)) AS fare_amount#11350, coalesce(nanvl(extra#10824, cast(null as double)), cast(0.0 as double)) AS extra#11351, coalesce(nanvl(mta_tax#10825, cast(null as double)), cast(0.0 as double)) AS mta_tax#113

#Part 9

In [170]:
!pip install pyngrok

In [171]:
from pyngrok import ngrok

# Replace with your own token
ngrok.set_auth_token("3HMQSImSmPUEeOt69nhWMnmFEhU_5UzFpqqE7ns5E4LkMtnSJ")

public_url = ngrok.connect(4040)
print(public_url)

NgrokTunnel: "https://delegator-wasp-freedom.ngrok-free.dev" -> "http://localhost:4040"


In [172]:
df.count()
df.groupBy("payment_type").count().show()
df.orderBy("fare_amount").show(10)

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2298422|
|           3|  10649|
|           2| 422945|
|           4|  22879|
+------------+-------+

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
| 

In [173]:
query1 = spark.sql("""
SELECT payment_type, COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY payment_type
""")

query1.toPandas().to_csv("query1.csv", index=False)

In [174]:
query2 = spark.sql("""
SELECT VendorID, AVG(fare_amount) AS Avg_Fare
FROM taxi_trips
GROUP BY VendorID
""")

query2.toPandas().to_csv("query2.csv", index=False)

In [175]:
query3 = spark.sql("""
SELECT PULocationID, COUNT(*) AS Trips
FROM taxi_trips
GROUP BY PULocationID
ORDER BY Trips DESC
LIMIT 10
""")

query3.toPandas().to_csv("query3.csv", index=False)

In [176]:
import platform
import sys
from datetime import datetime

log = f"""
Execution Log
===============================

Student Name : Fatima Asif
Roll No      : 231980015
Date         : {datetime.now().strftime('%d-%m-%Y %H:%M:%S')}

Environment
-----------
Platform           : Google Colab
Operating System   : {platform.system()}
Python Version     : {sys.version.split()[0]}
Apache Spark       : {spark.version}

Execution Summary
-----------------
[✓] Part 1 - Environment Setup
[✓] Part 2 - Load Dataset
[✓] Part 3 - Exploratory Data Analysis
[✓] Part 4 - Data Cleaning
[✓] Part 5 - Spark Transformations
[✓] Part 6 - Spark SQL
[✓] Part 7 - Window Functions
[✓] Part 8 - Performance Optimization
[✓] Part 9 - Spark UI Analysis

Dataset Statistics
------------------
Total Records : {df.count()}
Total Columns : {len(df.columns)}

Status
------
Execution Completed Successfully.
"""

with open("execution_log.txt", "w") as f:
    f.write(log)

print("execution_log.txt created successfully!")

execution_log.txt created successfully!


In [177]:
from google.colab import files

files.download("execution_log.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>